In [243]:
import pandas as pd 
import numpy as np 

import scipy.stats as  stats 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler

In [244]:
df = pd.read_csv('placement.csv')

In [245]:
df.head(10)

,branch,college_tier,cgpa,backlogs,coding_skills,dsa_score,aptitude_score,communication_skills,ml_knowledge,system_design,internships,projects_count,certifications,hackathons,open_source_contributions,extracurriculars,placement_status,salary_package_lpa
0,ECE,Tier-3,6.70,0,7.6,4.4,49.5,3.7,6.4,0.3,1,4,4,3,2,1,1,14.75
1,Chemical,Tier-2,5.70,0,5.4,7.9,72.0,8.3,6.3,1.9,0,4,0,0,0,0,0,NaN
2,EE,Tier-2,7.19,0,5.6,6.8,79.1,7.4,4.4,5.2,1,3,2,1,2,0,1,19.06
3,CE,Tier-2,6.48,0,5.2,3.1,48.4,5.0,1.1,6.7,1,4,3,0,0,0,0,NaN
4,CSE,Tier-2,6.71,1,5.9,4.7,61.2,4.3,2.7,2.8,1,2,0,3,0,1,1,13.42
5,CSE,Tier-3,6.21,1,6.7,6.5,69.9,5.0,4.0,5.2,2,3,3,2,0,1,1,16.47
6,CSE,Tier-3,7.91,0,7.0,3.9,61.5,6.7,4.9,2.2,2,5,2,1,0,3,1,16.51
7,IT,Tier-3,6.95,0,6.1,6.0,44.4,5.5,4.5,6.3,1,2,1,1,1,0,0,NaN
8,CE,Tier-2,7.03,0,5.1,2.5,54.7,6.4,6.2,3.2,1,3,0,3,0,1,1,13.49
9,EE,Tier-2,7.34,3,4.7,6.2,78.8,5.9,6.7,5.2,1,2,1,1,0,0,1,17.95


In [246]:
df.isnull().sum()

branch                           0
college_tier                     0
cgpa                             0
backlogs                         0
coding_skills                    0
dsa_score                        0
aptitude_score                   0
communication_skills             0
ml_knowledge                     0
system_design                    0
internships                      0
projects_count                   0
certifications                   0
hackathons                       0
open_source_contributions        0
extracurriculars                 0
placement_status                 0
salary_package_lpa           31525
dtype: int64

In [247]:
df.shape

(100000, 18)

In [248]:
df = df.drop(columns=['salary_package_lpa'])


In [249]:
df.head(5)

,branch,college_tier,cgpa,backlogs,coding_skills,dsa_score,aptitude_score,communication_skills,ml_knowledge,system_design,internships,projects_count,certifications,hackathons,open_source_contributions,extracurriculars,placement_status
0,ECE,Tier-3,6.70,0,7.6,4.4,49.5,3.7,6.4,0.3,1,4,4,3,2,1,1
1,Chemical,Tier-2,5.70,0,5.4,7.9,72.0,8.3,6.3,1.9,0,4,0,0,0,0,0
2,EE,Tier-2,7.19,0,5.6,6.8,79.1,7.4,4.4,5.2,1,3,2,1,2,0,1
3,CE,Tier-2,6.48,0,5.2,3.1,48.4,5.0,1.1,6.7,1,4,3,0,0,0,0
4,CSE,Tier-2,6.71,1,5.9,4.7,61.2,4.3,2.7,2.8,1,2,0,3,0,1,1


In [250]:
small_df = df.sample(n=5000, random_state=42)

In [251]:
X = df.iloc[:,0:16]   # features
y = df.iloc[:,16]     # target

In [252]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [253]:
X_train.sample(5)

,branch,college_tier,cgpa,backlogs,coding_skills,dsa_score,aptitude_score,communication_skills,ml_knowledge,system_design,internships,projects_count,certifications,hackathons,open_source_contributions,extracurriculars
23667,CSE,Tier-3,8.52,2,6.8,3.9,81.8,3.4,7.6,4.4,1,2,3,3,0,1
12515,ME,Tier-2,7.02,0,5.2,6.6,64.8,4.2,4.7,3.9,3,1,3,0,0,0
38846,ECE,Tier-2,6.95,0,5.5,6.4,66.5,7.2,3.8,3.7,1,1,1,1,0,0
51706,EE,Tier-2,7.76,1,7.5,2.2,59.5,5.2,4.9,2.7,1,1,2,0,0,0
38724,EE,Tier-2,5.30,0,5.3,6.5,75.8,3.7,2.9,2.8,2,2,2,1,0,0


In [254]:
X_train.shape

(80000, 16)

Your dataset has two types of features:

## Categorical Features

These contain text values.

branch → ECE, ME, IT, CE

college_tier → Tier-1, Tier-2, Tier-3

## Numerical Features

These already contain numbers.

cgpa

backlogs

coding_skills

dsa_score

aptitude_score

communication_skills

ml_knowledge

system_design

internships

projects_count

certifications

hackathons

open_source_contributions

In [255]:
preprocess = ColumnTransformer([
    
    ('branch_ohe',
     OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
     [0]),

    ('college_tier_oe',
     OrdinalEncoder(categories=[['Tier-1','Tier-2','Tier-3']],
                    handle_unknown='use_encoded_value',
                    unknown_value=-1),
     [1]),

    ('scale_numeric',
     MinMaxScaler(),
     slice(2,16))   

])



In [256]:
trf1

ColumnTransformer(remainder='passthrough',
                  transformers=[('ohe_branch',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 [0])])

In [257]:
pipe = Pipeline([
    ('preprocess',preprocess),
    ('model', RandomForestClassifier())
])

In [258]:
print(X_train.dtypes)

branch                        object
college_tier                  object
cgpa                         float64
backlogs                       int64
coding_skills                float64
dsa_score                    float64
aptitude_score               float64
communication_skills         float64
ml_knowledge                 float64
system_design                float64
internships                    int64
projects_count                 int64
certifications                 int64
hackathons                     int64
open_source_contributions      int64
extracurriculars               int64
dtype: object


In [259]:
print(df['college_tier'].value_counts())

college_tier
Tier-3    45222
Tier-2    39910
Tier-1    14868
Name: count, dtype: int64


In [260]:
df.groupby('branch')['placement_status'].mean()

branch
CE          0.646648
CSE         0.713487
Chemical    0.645532
ECE         0.695763
EE          0.663497
IT          0.708932
ME          0.664057
Name: placement_status, dtype: float64

In [261]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('branch_ohe',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [0]),
                                                 ('college_tier_oe',
                                                  OrdinalEncoder(categories=[['Tier-1',
                                                                              'Tier-2',
                                                                              'Tier-3']],
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  [1]),
                                                 ('scale_numeric',
                                                  MinMaxScaler(),
                                                  slice(2, 16, None))])),
                ('model', RandomForestClassifier())])

In [262]:
y_pred = pipe.predict(X_test)

In [263]:

accuracy_score(y_test, y_pred)

0.6921